In [ ]:
from pathlib import Path
import random

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, log_loss
from sklearn.pipeline import Pipeline

In [ ]:
market = pd.read_csv("marketdata_to_events.csv")

required = {"Date", "Open", "High", "Low", "Close", "Volume", "window", 
            "event_id", "symbol", "earnings_timestamp"}
assert required.issubset(market.columns)

market["Date"] = pd.to_datetime(market["Date"])
market["earnings_timestamp"] = pd.to_datetime(market["earnings_timestamp"])
market["event_id"] = market["event_id"].astype("int64")

assert not market.duplicated(["event_id", "Date"]).any()
market = market.sort_values(["event_id", "Date"])

files = {"train": "data/training_events.csv",
         "val": "data/validation_events.csv",
         "test": "data/testing_events.csv"}

splits = {}

for name, filename in files.items():
    frame = pd.read_csv(filename)
    frame["date"] = pd.to_datetime(frame["date"])
    frame["event_id"] = frame["event_id"].astype("int64")

    assert not frame["event_id"].duplicated().any()

    splits[name] = frame.sort_values(["date", "event_id"]).reset_index(drop=True)

all_ids = pd.concat([frame["event_id"] for frame in splits.values()], ignore_index=True)

assert not all_ids.duplicated().any()
wanted_ids = set(all_ids)

print("Market rows:", len(market))
print({name: len(frame) for name, frame in splits.items()})

## Daily sequence feature engineering

In [ ]:
daily_cols = ["close_return", "overnight_gap", "intraday_return", "high_low_range", 
              "close_location", "relative_volume", "volume_change"]
SEED = 42
HORIZON = 3
PRE_DAYS = 30

def build_daily_features(event):
    event = event.sort_values("Date")

    pre = event.loc[event["window"] == "pre"].tail(PRE_DAYS).copy()
    post = event.loc[event["window"] == "post"].copy()

    if len(pre) != PRE_DAYS or len(post) < HORIZON:
        raise ValueError("Insufficient trading sessions")

    prices = pre[["Open", "High", "Low", "Close"]].to_numpy()
    volume = pre["Volume"].to_numpy()

    if not np.isfinite(prices).all() or (prices <= 0).any():
        raise ValueError("Invalid prices")

    if not np.isfinite(volume).all() or (volume < 0).any():
        raise ValueError("Invalid volume")

    if (pre["High"] < pre["Low"]).any():
        raise ValueError("High below Low")

    previous_close = pre["Close"].shift(1)

    # Each reference uses only earlier sessions.
    previous_volume_mean = (
        pre["Volume"]
        .shift(1)
        .rolling(20, min_periods=1)
        .mean()
    )

    daily = pd.DataFrame(index=pre.index)

    daily["close_return"] = np.log(
        pre["Close"] / previous_close
    )

    daily["overnight_gap"] = np.log(
        pre["Open"] / previous_close
    )

    daily["intraday_return"] = np.log(
        pre["Close"] / pre["Open"]
    )

    daily["high_low_range"] = np.log(
        pre["High"] / pre["Low"]
    )

    price_range = pre["High"] - pre["Low"]

    daily["close_location"] = (
        (pre["Close"] - pre["Low"])
        / price_range.replace(0, np.nan)
    ).fillna(0.5)

    daily["relative_volume"] = np.log(
        (pre["Volume"] + 1) / (previous_volume_mean + 1)
    )

    daily["volume_change"] = np.log(
        (pre["Volume"] + 1)
        / (pre["Volume"].shift(1) + 1)
    )

    # First row has no previous close or previous volume.
    daily = daily.iloc[1:].copy()
    sequence = daily[daily_cols].to_numpy(dtype=np.float32)

    assert sequence.shape == (PRE_DAYS - 1, len(daily_cols))
    assert np.isfinite(sequence).all()

    features = {}
    closes = pre["Close"].to_numpy()

    for window in [5, 10, 20, PRE_DAYS - 1]:
        tail = daily.tail(window)

        features[f"momentum_{window}"] = np.expm1(
            tail["close_return"].sum()
        )

        features[f"volatility_{window}"] = (
            tail["close_return"].std()
        )

        features[f"range_mean_{window}"] = (
            tail["high_low_range"].mean()
        )

        features[f"relative_volume_mean_{window}"] = (
            tail["relative_volume"].mean()
        )

        features[f"overnight_mean_{window}"] = (
            tail["overnight_gap"].mean()
        )

        features[f"intraday_mean_{window}"] = (
            tail["intraday_return"].mean()
        )

        path = closes[-(window + 1):]
        features[f"max_drawdown_{window}"] = (
            path / np.maximum.accumulate(path) - 1
        ).min()

    features["volatility_ratio_5_20"] = (
        features["volatility_5"]
        / (features["volatility_20"] + 1e-8)
    )

    features["volume_ratio_5_20"] = (
        (pre["Volume"].tail(5).mean() + 1)
        / (pre["Volume"].tail(20).mean() + 1)
    )

    assert np.isfinite(list(features.values())).all()

    # Post-event rows are used only for target checks and purging.
    target_close = float(post["Close"].iloc[HORIZON - 1])

    if not np.isfinite(target_close) or target_close <= 0:
        raise ValueError("Invalid target close")

    expected_return = (
        target_close / float(pre["Close"].iloc[-1]) - 1
    )

    metadata = {
        "event_id": int(event["event_id"].iloc[0]),
        "market_symbol": event["symbol"].iloc[0],
        "market_timestamp": event["earnings_timestamp"].iloc[0],
        "label_end": post["Date"].iloc[HORIZON - 1],
        "expected_return": expected_return
    }

    return sequence, features, metadata


sequences = {}
feature_rows = []

selected_market = market.loc[market["event_id"].isin(wanted_ids)]

for event_id, event in selected_market.groupby("event_id"):
    sequence, features, metadata = build_daily_features(event)
    sequences[int(event_id)] = sequence
    feature_rows.append({**metadata, **features})

feature_table = pd.DataFrame(feature_rows)

metadata_cols = {
    "event_id", "market_symbol", "market_timestamp",
    "label_end", "expected_return"
}

FEATURE_COLS = [col for col in feature_table.columns if col not in metadata_cols]

print("Usable events:", len(sequences))
print("Engineered features:", len(FEATURE_COLS))

feature_table.head()

In [ ]:
# 
feature_table.columns

In [ ]:
# Market time series data
len(sequences), sequences[0].shape

## Build CNN inputs from sequences

In [ ]:
SEED = 42
LABEL_COL = "label_3d"

BATCH_SIZE = 128
MAX_EPOCHS = 40
PATIENCE = 6
LEARNING_RATE = 1e-3
EMBEDDING_DIM = 8
DROPOUT = 0.25
N_COMPONENTS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

In [ ]:
raw_sequences = {}
X_sequence = {}
labels = {}
event_ids = {}

files = {"train": "data/training_events.csv",
         "val": "data/validation_events.csv",
         "test": "data/testing_events.csv"}

for name, filename in files.items():
    frame = pd.read_csv(filename)
    event_ids[name] = frame["event_id"].to_numpy(dtype=np.int64)

    # Match the original split's row order.
    raw_sequences[name] = np.stack([sequences[int(event_id)] for event_id in event_ids[name]]).astype(np.float32)
    labels[name] = frame[LABEL_COL].to_numpy(dtype=np.float32)

    assert np.isfinite(raw_sequences[name]).all()

sequence_length = raw_sequences["train"].shape[1]
n_channels = raw_sequences["train"].shape[2]

for name in files:
    assert raw_sequences[name].shape[1:] == (sequence_length, n_channels)

# Fit channel scaling on training data only.
sequence_scaler = StandardScaler().fit(raw_sequences["train"].reshape(-1, n_channels))

for name, raw in raw_sequences.items():
    scaled = sequence_scaler.transform(raw.reshape(-1, n_channels)).reshape(raw.shape)
    X_sequence[name] = np.ascontiguousarray(scaled.transpose(0, 2, 1), dtype=np.float32)
    print(name, X_sequence[name].shape)

# Create a batch load
def make_sequence_loader(name, shuffle=False):
    dataset = TensorDataset(torch.from_numpy(X_sequence[name]), torch.from_numpy(labels[name]))

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        generator=torch.Generator().manual_seed(SEED)
    )


loaders = {
    name: make_sequence_loader(name, shuffle=(name == "train"))
    for name in files
}

joblib.dump(sequence_scaler, "data/cnn/sequence_scaler.joblib")

In [ ]:
class DailySequenceCNN(nn.Module):
    def __init__(self, n_channels, embedding_dim=4, dropout=DROPOUT):
        super().__init__()

        # extra_layers = []
        # for _ in range(6):
        #     extra_layers.extend([
        #         nn.Conv1d(
        #             embedding_dim,
        #             embedding_dim,
        #             kernel_size=1
        #         ),
        #         nn.ReLU()
        #     ])

        self.encoder = nn.Sequential(
            nn.Conv1d(
                n_channels, 16,
                kernel_size=3, padding=1
            ),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(
                16, 16,
                kernel_size=3, padding=2, dilation=2
            ),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(
                16, 16,
                kernel_size=3, padding=4, dilation=4
            ),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(
                16, embedding_dim,
                kernel_size=3, padding=8, dilation=8
            ),
            nn.BatchNorm1d(8),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            # *extra_layers,

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten()

        )

        # Used only to train the CNN.
        self.classifier = nn.Linear(embedding_dim, 1)

    def forward(self, daily):
        embedding = self.encoder(daily)
        return self.classifier(embedding).squeeze(-1)


cnn = DailySequenceCNN(
    n_channels=n_channels,
    embedding_dim=EMBEDDING_DIM,
    dropout=DROPOUT
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    cnn.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-3
)

print(cnn)

In [ ]:
@torch.no_grad()
def evaluate_cnn(loader):
    cnn.eval()

    total_loss = 0.0
    count = 0
    probabilities = []
    targets = []

    for daily, target in loader:
        daily = daily.to(DEVICE)
        target = target.to(DEVICE)

        logits = cnn(daily)
        loss = criterion(logits, target)

        total_loss += loss.item() * len(target)
        count += len(target)

        probabilities.append(
            torch.sigmoid(logits).cpu().numpy()
        )
        targets.append(target.cpu().numpy())

    return (
        total_loss / count,
        np.concatenate(probabilities),
        np.concatenate(targets)
    )


history = []
best_loss = float("inf")
best_state = None
best_epoch = None
stale_epochs = 0

for epoch in range(1, MAX_EPOCHS + 1):
    cnn.train()

    total_loss = 0.0
    count = 0

    for daily, target in loaders["train"]:
        daily = daily.to(DEVICE)
        target = target.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits = cnn(daily)
        loss = criterion(logits, target)

        assert torch.isfinite(loss)

        loss.backward()
        nn.utils.clip_grad_norm_(cnn.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(target)
        count += len(target)

    val_loss, val_probability, val_target = evaluate_cnn(loaders["val"])
    val_auc = roc_auc_score(val_target, val_probability)

    history.append({
        "epoch": epoch,
        "train_loss": total_loss / count,
        "val_loss": val_loss,
        "val_auc": val_auc
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {total_loss / count:.4f} | "
        f"val loss {val_loss:.4f} | "
        f"val AUC {val_auc:.4f}"
    )

    if val_loss < best_loss - 1e-4:
        best_loss = val_loss
        best_epoch = epoch
        stale_epochs = 0

        best_state = {
            key: value.detach().cpu().clone()
            for key, value in cnn.state_dict().items()
        }
    else:
        stale_epochs += 1

        if stale_epochs >= PATIENCE:
            print("Early stopping")
            break

assert best_state is not None
cnn.load_state_dict(best_state)
cnn.eval()

torch.save(
    {
        "state_dict": best_state,
        "n_channels": n_channels,
        "sequence_length": sequence_length,
        "embedding_dim": EMBEDDING_DIM,
        "dropout": DROPOUT,
        "label_col": LABEL_COL,
        "best_epoch": best_epoch
    },
    "data/cnn/daily_sequence_cnn.pt"
)

pd.DataFrame(history).to_csv(
    "data/cnn/cnn_training_history.csv",
    index=False
)

print("Restored best epoch:", best_epoch)

## Daily sequence and transcript emebeddings cocatenation using MLP for prediction

In [ ]:
ROOT = Path("/Users/cczh/Desktop/PEAD-Prediction")
OUT = ROOT / "data" / "mlp_fusion"
OUT.mkdir(parents=True, exist_ok=True)

LABEL_COL = "label_3d"

split_files = {
    "train": "training_events.csv",
    "val": "validation_events.csv",
    "test": "testing_events.csv"
}

fusion_splits = {
    name: pd.read_csv(ROOT / "data" / filename)
    for name, filename in split_files.items()
}

# Check that text rows match the CNN input order.
for name, frame in fusion_splits.items():
    np.testing.assert_array_equal(
        frame["event_id"].to_numpy(),
        event_ids[name]
    )

    assert len(frame) == len(X_sequence[name])

# Extract embeddings with dropout disabled.
cnn.to(DEVICE)
cnn.eval()

market_embeddings = {}

with torch.no_grad():
    for name in fusion_splits:
        batches = []

        for daily, _ in make_sequence_loader(name, shuffle=False):
            embedding = cnn.encoder(daily.to(DEVICE))
            batches.append(embedding.cpu().numpy())

        market_embeddings[name] = np.concatenate(batches)

# Fit text PCA on training data only.
embedding_cols = [f"embedding_{i}" for i in range(768)]

for frame in fusion_splits.values():
    assert np.isfinite(frame[embedding_cols].to_numpy()).all()

text_pca = PCA(n_components=20, random_state=42)

text_pca.fit(
    fusion_splits["train"][embedding_cols].to_numpy()
)

text_embeddings = {
    name: text_pca.transform(frame[embedding_cols].to_numpy())
    for name, frame in fusion_splits.items()
}

joblib.dump(text_pca, OUT / "text_pca_5.joblib")

print(
    "Text variance preserved:",
    f"{text_pca.explained_variance_ratio_.sum():.2%}"
)

In [ ]:
cnn_dim = market_embeddings["train"].shape[1]

cnn_cols = [f"cnn_market_{i}" for i in range(cnn_dim)]
text_cols = [f"text_pc_{i}" for i in range(20)]
FUSION_COLS = cnn_cols + text_cols

fusion_features = {}
fusion_labels = {}

for name, frame in fusion_splits.items():
    fusion_features[name] = np.concatenate(
        [
            market_embeddings[name],
            text_embeddings[name]
        ],
        axis=1
    ).astype(np.float32)

    fusion_labels[name] = frame[
        LABEL_COL
    ].to_numpy(dtype=np.float32)

    assert np.isfinite(fusion_features[name]).all()
    assert frame[LABEL_COL].isin([0, 1]).all()

    metadata = frame[
        ["event_id", "symbol", "date", LABEL_COL]
    ].reset_index(drop=True)

    features = pd.DataFrame(
        fusion_features[name],
        columns=FUSION_COLS
    )

    dataset = pd.concat([metadata, features], axis=1)

    dataset.to_csv(
        OUT / f"{name}_fusion_features.csv",
        index=False
    )

    # Also save the CNN embedding separately.
    np.savez_compressed(
        OUT / f"{name}_cnn_embeddings.npz",
        event_id=frame["event_id"].to_numpy(),
        embeddings=market_embeddings[name]
    )

    print(name, fusion_features[name].shape)

joblib.dump(FUSION_COLS, OUT / "fusion_columns.joblib")

## Logistic Regression with Time Sequence Embeddings and PCA Text Embeddings

In [ ]:
ROOT = Path("/Users/cczh/Desktop/PEAD-Prediction")
DATA_DIR = ROOT / "data" / "mlp_fusion"
LR_OUT = ROOT / "data" / "lr_fusion"
LR_OUT.mkdir(parents=True, exist_ok=True)

LABEL_COL = "label_3d"

datasets = {
    name: pd.read_csv(DATA_DIR / f"{name}_fusion_features.csv")
    for name in ["train", "val", "test"]
}

# Select only model inputs, excluding metadata and targets.
cnn_cols = sorted(
    [
        col for col in datasets["train"].columns
        if col.startswith("cnn_market_")
    ],
    key=lambda col: int(col.rsplit("_", 1)[1])
)

text_cols = [f"text_pc_{i}" for i in range(20)]
FEATURE_COLS = cnn_cols + text_cols

assert cnn_cols, "No CNN embedding columns found."

X = {}
y = {}

for name, frame in datasets.items():
    X[name] = frame[FEATURE_COLS].to_numpy(dtype=np.float64)
    y[name] = frame[LABEL_COL].to_numpy(dtype=np.int64)

    assert np.isfinite(X[name]).all()
    assert frame[LABEL_COL].isin([0, 1]).all()
    assert not frame["event_id"].duplicated().any()

    print(name, X[name].shape)

print("CNN dimensions:", len(cnn_cols))
print("Text dimensions:", len(text_cols))
print("Total LR features:", len(FEATURE_COLS))

In [ ]:
C_VALUES = [0.001, 0.01, 0.1, 1.0, 10.0]

validation_results = []
best_lr = None
best_C = None
best_val_loss = float("inf")

for C in C_VALUES:
    candidate = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            C=C,
            max_iter=3000,
            random_state=42
        ))
    ])

    candidate.fit(X["train"], y["train"])

    val_probability = candidate.predict_proba(
        X["val"]
    )[:, 1]

    val_loss = log_loss(
        y["val"], val_probability, labels=[0, 1]
    )

    validation_results.append({
        "C": C,
        "val_log_loss": val_loss,
        "val_auc": roc_auc_score(
            y["val"], val_probability
        ),
        "val_accuracy": accuracy_score(
            y["val"], val_probability >= 0.5
        )
    })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_C = C
        best_lr = candidate

validation_results = pd.DataFrame(validation_results)

print(validation_results.round(5).to_string(index=False))
print("Selected C:", best_C)